# Tutorial: Palette catalog summaries

**Audience:** Python beginners who want to summarize rows from a CSV catalog.

**Prerequisites:** Basic functions, dictionaries, lists, and file paths. Only the Python standard library is used.

**Learning goals:** Read a UTF-8 CSV by header name, normalize grouping values, count every row, collect unique product types, and sort a summary deterministically.


## Outline

1. Import the CSV reader.
2. Define a reusable grouping function.
3. Summarize the public palette catalog.
4. Check important edge cases.
5. Practice with exercises.


In [ ]:
from __future__ import annotations

import csv
import tempfile
from pathlib import Path


## Build the summary function

`csv.DictReader` uses the header row, so column order does not matter. For each data row, the primary color and product type are stripped; the color is also uppercased. A set keeps product types unique, while a separate counter still includes duplicate rows.


In [ ]:
def summarize_palettes(csv_path: str | Path) -> list[dict[str, object]]:
    """Group catalog rows by normalized primary color."""
    groups: dict[str, dict[str, object]] = {}

    with open(csv_path, "r", encoding="utf-8", newline="") as handle:
        for row in csv.DictReader(handle):
            primary = row["Primary (Hex)"].strip().upper()
            product_type = row["Product Type"].strip()
            group = groups.setdefault(
                primary, {"count": 0, "product_types": set()}
            )
            group["count"] += 1
            group["product_types"].add(product_type)

    ordered_groups = sorted(
        groups.items(), key=lambda item: (-item[1]["count"], item[0])
    )
    return [
        {
            "primary": primary,
            "count": group["count"],
            "product_types": sorted(group["product_types"]),
        }
        for primary, group in ordered_groups
    ]


## Summarize the catalog

The next cell applies the function to the supplied file. The first few objects provide a compact preview; each object contains only the normalized primary color, its row count, and sorted unique product types.


In [ ]:
palette_summary = summarize_palettes('/inputs/colors.csv')
palette_summary[:5]


## Check edge cases

This small schema-compatible catalog changes the column order, repeats a row, uses lowercase color text, and includes a quoted product name containing a comma. A second file checks that headers without data produce an empty list.


In [ ]:
sample_csv = (
    "Product Type,Primary (Hex),Notes\n"
    "  Studio  ,  #aabbcc  ,first\n"
    "  Studio  ,#AABBCC,duplicate\n"
    "\"Shop, Retail\",#001122,quoted comma\n"
)

with tempfile.TemporaryDirectory() as temporary_directory:
    sample_path = Path(temporary_directory) / "sample.csv"
    empty_path = Path(temporary_directory) / "empty.csv"
    sample_path.write_text(sample_csv, encoding="utf-8")
    empty_path.write_text("Product Type,Primary (Hex),Notes\n", encoding="utf-8")

    assert summarize_palettes(sample_path) == [
        {"primary": "#AABBCC", "count": 2, "product_types": ["Studio"]},
        {"primary": "#001122", "count": 1, "product_types": ["Shop, Retail"]},
    ]
    assert summarize_palettes(empty_path) == []


## Exercises

1. Write a tiny catalog where two colors have equal counts and predict their lexicographic order.
2. Add repeated product types to a group and confirm that `count` grows while `product_types` stays unique.
3. Adapt the approach to group by `Secondary (Hex)` without changing the sorting rules.
